# Time Series Forecasting with Ridge Regression on Lag Features

## What it does
Constructs a **lag-feature matrix** from one or more time series and forecasts a target variable
`h` steps ahead using **Ridge regression** with cross-validated hyperparameter selection.

The pipeline treats forecasting as a supervised learning problem:
- **Input (X):** lags 1–`MAX_LAG` of all selected predictors
- **Output (y):** target series shifted `HORIZON` steps into the future
- **Regularization:** Ridge (L2) penalizes noisy lag coefficients, avoids overfitting in high-lag settings

## When to use it
- You have a stationary macro/financial time series and want to forecast it `h` steps ahead
- You have many potential predictors (e.g. FRED-MD) and want regularized selection
- You need a strong ML baseline before moving to more complex models (GBR, LSTM, etc.)

## Data format
A wide CSV or Parquet file with rows = time periods and columns = variables.
The `TARGET_COL` column is the series to forecast; all other numeric columns are used as predictors.
A `DATE_COL` column is used for chronological splitting — no random shuffling.

## Evaluation metrics
| Metric | Description |
|---|---|
| RMSE | Root Mean Squared Error — penalizes large errors |
| MAE | Mean Absolute Error — robust to outliers |
| OOS R² | Out-of-sample R² using training mean as benchmark (Campbell & Thompson 2008) |
| Dir. Acc. | Directional accuracy — fraction of periods where sign(forecast) = sign(actual) |

## Configuration

Edit the values below to adapt this notebook to a new dataset.

In [ ]:
CONFIG = {
    # --- Data ---
    'DATA_FILE':          '../../data/FREDMD.csv',
    'DATE_COL':           'sasdate',
    # --- Target to forecast ---
    'TARGET_COL':         'INDPRO',     # industrial production index
    # --- Forecast horizon (steps ahead) ---
    'HORIZON':            1,
    # --- Lag features ---
    'MAX_LAG':            12,           # include lags 1 through MAX_LAG
    # --- Time splits (year-based) ---
    'TRAIN_YEARS':        20,
    'VAL_YEARS':          10,
    # --- Preprocessing ---
    'MIN_OBS_FRAC':       0.5,          # drop columns with < this fraction of non-null obs
    'STANDARDIZE':        True,
    # --- Ridge grid ---
    'RIDGE_ALPHAS':       [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
    # --- Output ---
    'SAVE_RESULTS':       True,
    'OUTPUT_DIR':         '../../results',
}

print('Configuration loaded.')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## Step 1 — Load Data & Clean

In [ ]:
import sys, warnings, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('../../').resolve()))
from utils import load_csv, load_parquet, compute_oos_r2

if CONFIG['DATA_FILE'].endswith(('.pq', '.parquet')):
    df_raw = load_parquet(CONFIG['DATA_FILE'])
else:
    df_raw = load_csv(CONFIG['DATA_FILE'])

# Set date as index
if CONFIG['DATE_COL'] in df_raw.columns:
    df_raw[CONFIG['DATE_COL']] = pd.to_datetime(df_raw[CONFIG['DATE_COL']])
    df_raw = df_raw.set_index(CONFIG['DATE_COL']).sort_index()

# Keep only numeric columns, drop sparse ones
df_raw = df_raw.select_dtypes(include='number')
min_obs = int(len(df_raw) * CONFIG['MIN_OBS_FRAC'])
df_clean = df_raw.dropna(axis=1, thresh=min_obs)

assert CONFIG['TARGET_COL'] in df_clean.columns, \
    f"TARGET_COL '{CONFIG['TARGET_COL']}' not found. Available: {list(df_clean.columns[:10])}"

print(f'Raw shape     : {df_raw.shape}')
print(f'After cleaning: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns')
print(f'Period        : {df_clean.index[0].date()} — {df_clean.index[-1].date()}')
print(f'Target column : {CONFIG["TARGET_COL"]}')
df_clean.head()

## Step 2 — Build Lag Features & Time Splits

For each predictor column, we create lags 1 through `MAX_LAG`. The target is shifted `HORIZON` steps
into the future so that `y[t]` = value at time `t + HORIZON` and `X[t]` = features known at time `t`.

Splits are **chronological** — no shuffling, no look-ahead.

In [ ]:
# Build lag feature matrix
lag_frames = []
for lag in range(1, CONFIG['MAX_LAG'] + 1):
    shifted = df_clean.shift(lag)
    shifted.columns = [f'{c}_lag{lag}' for c in df_clean.columns]
    lag_frames.append(shifted)

X_all = pd.concat(lag_frames, axis=1)
y_all = df_clean[CONFIG['TARGET_COL']].shift(-CONFIG['HORIZON'])  # future target

# Align and drop NaNs introduced by shifting
combined = pd.concat([X_all, y_all.rename('__target__')], axis=1).dropna()
X_all = combined.drop(columns=['__target__'])
y_all = combined['__target__']

feature_cols = list(X_all.columns)
print(f'Lag features  : {len(feature_cols)} ({df_clean.shape[1]} series × {CONFIG["MAX_LAG"]} lags)')
print(f'Usable obs    : {len(y_all)}')

# Chronological train / val / test split
years = X_all.index.year
train_end_year = years.min() + CONFIG['TRAIN_YEARS'] - 1
val_end_year   = train_end_year + CONFIG['VAL_YEARS']

train_mask = years <= train_end_year
val_mask   = (years > train_end_year) & (years <= val_end_year)
test_mask  = years > val_end_year

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val,   y_val   = X_all[val_mask],   y_all[val_mask]
X_test,  y_test  = X_all[test_mask],  y_all[test_mask]

print(f'\nTrain : {X_train.index[0].date()} — {X_train.index[-1].date()}  ({len(y_train):,} obs)')
print(f'Val   : {X_val.index[0].date()} — {X_val.index[-1].date()}    ({len(y_val):,} obs)')
print(f'Test  : {X_test.index[0].date()} — {X_test.index[-1].date()}    ({len(y_test):,} obs)')

## Step 3 — Standardize Features

In [ ]:
if CONFIG['STANDARDIZE']:
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)
    X_test_s  = scaler.transform(X_test)
    print('Features standardized (fit on training data only).')
else:
    X_train_s, X_val_s, X_test_s = X_train.values, X_val.values, X_test.values
    print('Standardization skipped.')

y_train_mean = float(y_train.mean())
print(f'Training mean (benchmark): {y_train_mean:.6f}')

## Step 4 — Ridge Grid Search

Best alpha is selected by **validation RMSE** only. Test set is never touched during tuning.

In [ ]:
from sklearn.metrics import mean_squared_error

grid_results = []
for alpha in CONFIG['RIDGE_ALPHAS']:
    model = Ridge(alpha=alpha)
    model.fit(X_train_s, y_train)
    val_pred = model.predict(X_val_s)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    val_oos_r2 = compute_oos_r2(y_val.values, val_pred, y_train_mean)
    grid_results.append({'alpha': alpha, 'val_rmse': val_rmse, 'val_oos_r2': val_oos_r2, 'model': model})
    print(f'  alpha={alpha:>8.3f}  Val RMSE={val_rmse:.6f}  OOS R²={val_oos_r2*100:+.4f}%')

best = min(grid_results, key=lambda r: r['val_rmse'])
print(f"\nBest alpha={best['alpha']}  (Val RMSE={best['val_rmse']:.6f},  OOS R²={best['val_oos_r2']*100:+.4f}%)"
)

# Plot: RMSE vs alpha
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(
    [r['alpha'] for r in grid_results],
    [r['val_rmse'] for r in grid_results],
    'b-o', linewidth=2, markersize=7
)
ax.axvline(x=best['alpha'], color='red', linestyle='--', label=f"Best alpha={best['alpha']}")
ax.set(xlabel='Alpha (log scale)', ylabel='Val RMSE', title='Ridge Regression: RMSE vs Alpha')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5 — Final Evaluation on Test Set

In [ ]:
from sklearn.metrics import mean_absolute_error

best_model = best['model']

train_pred = best_model.predict(X_train_s)
val_pred   = best_model.predict(X_val_s)
test_pred  = best_model.predict(X_test_s)

def directional_accuracy(y_true, y_pred):
    return float(np.mean(np.sign(y_true) == np.sign(y_pred)))

metrics = {
    'rmse_train':    float(np.sqrt(mean_squared_error(y_train, train_pred))),
    'rmse_val':      float(np.sqrt(mean_squared_error(y_val,   val_pred))),
    'rmse_test':     float(np.sqrt(mean_squared_error(y_test,  test_pred))),
    'mae_test':      float(mean_absolute_error(y_test, test_pred)),
    'oos_r2_val':    float(compute_oos_r2(y_val.values,  val_pred,  y_train_mean)),
    'oos_r2_test':   float(compute_oos_r2(y_test.values, test_pred, y_train_mean)),
    'dir_acc_test':  directional_accuracy(y_test.values, test_pred),
}

print('RIDGE FORECASTING — TEST EVALUATION')
print('=' * 55)
print(f"  Best alpha       : {best['alpha']}")
print(f"  Horizon          : {CONFIG['HORIZON']} step(s) ahead")
print(f"  Max lag          : {CONFIG['MAX_LAG']}")
print(f"  Lag features     : {len(feature_cols)}")
print()
print(f"  Train RMSE       : {metrics['rmse_train']:.6f}")
print(f"  Val RMSE         : {metrics['rmse_val']:.6f}")
print(f"  Test RMSE        : {metrics['rmse_test']:.6f}")
print(f"  Test MAE         : {metrics['mae_test']:.6f}")
print()
print(f"  OOS R² Val       : {metrics['oos_r2_val']*100:+.4f}%")
print(f"  OOS R² Test      : {metrics['oos_r2_test']*100:+.4f}%")
print(f"  Dir. Accuracy    : {metrics['dir_acc_test']*100:.1f}%")
print('=' * 55)

## Step 6 — Actual vs Forecast Plot

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full series: actual vs fitted (train) and forecast (test)
axes[0].plot(y_train.index, y_train.values,      color='steelblue', linewidth=1,   label='Actual (train)', alpha=0.8)
axes[0].plot(y_train.index, train_pred,           color='steelblue', linewidth=1,   linestyle='--', alpha=0.5, label='Fitted (train)')
axes[0].plot(y_val.index,   y_val.values,         color='gray',      linewidth=1,   label='Actual (val)',   alpha=0.7)
axes[0].plot(y_test.index,  y_test.values,        color='black',     linewidth=1.5, label='Actual (test)')
axes[0].plot(y_test.index,  test_pred,            color='tomato',    linewidth=1.5, label='Forecast (test)')
axes[0].axvline(x=y_val.index[0],   color='gray', linestyle=':', linewidth=1)
axes[0].axvline(x=y_test.index[0],  color='gray', linestyle=':', linewidth=1)
axes[0].set(ylabel=CONFIG['TARGET_COL'],
            title=f'{CONFIG["TARGET_COL"]} — Ridge Forecast ({CONFIG["HORIZON"]}-step ahead, alpha={best["alpha"]})')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Forecast errors on test set
errors = y_test.values - test_pred
axes[1].bar(range(len(errors)), errors, color=np.where(errors >= 0, 'steelblue', 'tomato'), alpha=0.7)
axes[1].axhline(y=0, color='black', linewidth=0.8)
axes[1].set(xlabel='Test period', ylabel='Forecast error',
            title=f'Forecast Errors (Test)  RMSE={metrics["rmse_test"]:.4f}  MAE={metrics["mae_test"]:.4f}')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Step 7 — Top Predictive Lags

The Ridge coefficients (on standardized features) indicate which lags of which series matter most.

In [ ]:
top_n = 20
coef_df = pd.DataFrame({'feature': feature_cols, 'coefficient': best_model.coef_})
coef_df = coef_df.sort_values('coefficient', key=abs, ascending=False).head(top_n)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['steelblue' if v > 0 else 'tomato' for v in coef_df['coefficient']]
ax.barh(range(top_n), coef_df['coefficient'].values, color=colors, alpha=0.8)
ax.set_yticks(range(top_n))
ax.set_yticklabels(coef_df['feature'].values, fontsize=9)
ax.invert_yaxis()
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set(xlabel='Coefficient (standardized)', title=f'Top {top_n} Predictive Lags')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Step 8 — Save Results

In [ ]:
if CONFIG['SAVE_RESULTS']:
    os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

    summary = {
        'model':           'Ridge Forecasting',
        'target':          CONFIG['TARGET_COL'],
        'horizon':         CONFIG['HORIZON'],
        'max_lag':         CONFIG['MAX_LAG'],
        'best_alpha':      best['alpha'],
        'n_lag_features':  len(feature_cols),
        'rmse_train':      metrics['rmse_train'],
        'rmse_val':        metrics['rmse_val'],
        'rmse_test':       metrics['rmse_test'],
        'mae_test':        metrics['mae_test'],
        'oos_r2_val':      metrics['oos_r2_val'],
        'oos_r2_test':     metrics['oos_r2_test'],
        'dir_acc_test':    metrics['dir_acc_test'],
        'train_obs':       len(y_train),
        'val_obs':         len(y_val),
        'test_obs':        len(y_test),
        'standardized':    CONFIG['STANDARDIZE'],
        'notebook':        'forecasting',
    }

    path = os.path.join(CONFIG['OUTPUT_DIR'], 'forecasting_summary.csv')
    pd.DataFrame([summary]).to_csv(path, index=False)
    print(f'Saved: {path}')

    # Save test forecasts
    forecast_df = pd.DataFrame({
        'date':    y_test.index,
        'actual':  y_test.values,
        'forecast': test_pred,
        'error':   y_test.values - test_pred,
    })
    path = os.path.join(CONFIG['OUTPUT_DIR'], 'forecasting_test_predictions.csv')
    forecast_df.to_csv(path, index=False)
    print(f'Saved: {path}')
else:
    print('SAVE_RESULTS = False — skipping.')